# 02 — Predictive Model

**Catalog Campaign Profitability & Customer Targeting Analysis**

## Purpose

Build a model that estimates average sale amount for each of the 250 campaign
prospects, so that expected revenue can be calculated in notebook 03.

## Why a model at all?

The company knows what its existing customers spend. It does not know what these
250 prospects will spend, because they have no catalog purchase history. The
model transfers the relationship learned from 2,375 known customers onto the
prospects using characteristics both groups share.

## Why linear regression?

The business question is not "what is the most accurate possible prediction". It
is **"which customer characteristics move average sale amount, and by how much in
dollars"**. Linear regression answers both at once:

| Reason | Detail |
|---|---|
| **Interpretability** | Each coefficient is a dollar amount a marketing stakeholder can read directly. "A dual-relationship customer is worth $282 more than a credit-card-only customer" is actionable in a way a tree ensemble's feature importance is not. |
| **The relationship is linear** | EDA showed a near-linear relationship between products purchased and sale amount (r = 0.86). Flexibility would buy little. |
| **Small, low-dimensional feature set** | Two predictors and four segment levels. There is no complex interaction structure for a flexible model to discover. |
| **Auditability** | Finance can reproduce any prediction with a calculator. That matters when the output justifies budget. |
| **Continuous target** | Average sale amount is a dollar quantity, so this is a regression problem, not a classification one. |

A benchmark against two more flexible models is run at the end of this notebook
to test whether that choice costs anything material.

**Requirements addressed:** BR-005, BR-012 / FR-004 to FR-009.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src import data_cleaning as dc
from src import evaluation as ev
from src import feature_engineering as fe
from src import modeling as md

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

customers = dc.clean_customers(dc.load_customers())
print(f"Training data: {len(customers):,} customers")

## 1. Target variable

`Avg_Sale_Amount` — the average dollar value of a customer's purchases.

**Why this target rather than, say, total revenue or response probability?**

- The campaign's revenue from a responder is the size of the order they place.
  Average sale amount is the best available estimate of that.
- Response probability is already supplied with the mailing list (`Score_Yes`),
  so it does not need to be modelled.
- Together, the two answer the business question: expected revenue is *what they
  spend if they buy* multiplied by *whether they buy*.

## 2. Predictor selection

Selection is driven by the business rules in
[`docs/business_rules.md`](../docs/business_rules.md), not by a search over
combinations.

In [ ]:
selection = pd.DataFrame([
    {"Variable": "Avg_Num_Products_Purchased", "Decision": "INCLUDE",
     "Rationale": "Strongest driver (r = 0.86); available in both files"},
    {"Variable": "Customer_Segment", "Decision": "INCLUDE",
     "Rationale": "Segments differ by up to 7x in mean sale; independent of basket size"},
    {"Variable": "Years_as_Customer", "Decision": "EXCLUDE",
     "Rationale": "r = 0.03 with target; inconsistent type and scale across files (C-05)"},
    {"Variable": "Responded_to_Last_Catalog", "Decision": "EXCLUDE",
     "Rationale": "Absent from the mailing list - model would be unscoreable (BRule-06)"},
    {"Variable": "Customer_ID / Store_Number / ZIP", "Decision": "EXCLUDE",
     "Rationale": "Numeric labels, not quantities (BRule-05)"},
    {"Variable": "Name / Address / City / State", "Decision": "EXCLUDE",
     "Rationale": "Personally identifying; no predictive content (BRule-05, NFR-007)"},
])
selection

### Categorical encoding

`Customer_Segment` is one-hot encoded with **`Credit Card Only` as the reference
level**. One level must be dropped to avoid the dummy-variable trap — if all four
indicators were included they would sum to 1 for every row and be perfectly
collinear with the intercept.

Fixing the reference level means the coefficients keep the same meaning across
model refreshes: each is the dollar difference versus an otherwise identical
Credit-Card-Only customer.

In [ ]:
X = fe.build_feature_matrix(customers)
y = customers["Avg_Sale_Amount"]

print("Design matrix columns:")
for col in X.columns:
    print(f"  - {col}")
print(f"\nShape: {X.shape[0]:,} rows x {X.shape[1]} features")
X.head()

## 3. Train / test split

The source project fitted on the full dataset. That is a reasonable production
choice — more data gives better coefficient estimates — but it cannot tell you
whether the model generalises. Here both are done:

- A **held-out split** (75/25) gives an honest estimate of out-of-sample performance.
- **5-fold cross-validation** checks that the estimate is stable rather than an artefact of one lucky split.
- The **final model** is refitted on all 2,375 records for scoring.

In [ ]:
holdout = md.holdout_evaluation(customers)

split_table = pd.DataFrame([
    {"Split": "Train", "n": holdout["n_train"], **{k: v for k, v in holdout["train"].items() if k != "n"}},
    {"Split": "Test", "n": holdout["n_test"], **{k: v for k, v in holdout["test"].items() if k != "n"}},
]).rename(columns={"r2": "R-squared", "mae": "MAE ($)", "rmse": "RMSE ($)", "mape_pct": "MAPE (%)"})
split_table

In [ ]:
cv = md.cross_validated_r2(customers, folds=5)
print(f"5-fold cross-validated R-squared: {cv['mean_r2']:.4f} (std {cv['std_r2']:.4f})")
print("Fold scores: " + ", ".join(f"{s:.4f}" for s in cv["scores"]))

### Interpretation

Train R-squared (0.839) and test R-squared (0.832) are almost identical, and the
five cross-validation folds range narrowly around 0.835 with a standard deviation
of 0.015.

**The model is not overfitting.** It performs the same on data it has never seen
as on data it was fitted to, which is what you would expect from a two-predictor
linear model on 2,375 records. The reported performance can be trusted as a
description of how the model will behave on the 250 prospects.

## 4. Final model

Refitted on the full dataset for scoring.

In [ ]:
model = md.fit_linear_model(customers)
coefs = model.coefficients

coef_table = pd.DataFrame({
    "Term": coefs.index,
    "Coefficient ($)": coefs.values,
}).sort_values("Coefficient ($)", ascending=False).reset_index(drop=True)
coef_table

In [ ]:
fitted = model.predict(customers)
insample = ev.regression_metrics(y, fitted)

print("Fitted model performance (full training data):")
for k, v in insample.items():
    print(f"  {k:>10}: {v:,.4f}")
print()
print(ev.interpret_r2(insample["r2"]))

### The model in plain English

```
Predicted Average Sale Amount =
      $303.46                                         (baseline)
    + $66.98  x  Average Number of Products Purchased
    + $281.84  if  Loyalty Club and Credit Card
    - $149.36  if  Loyalty Club Only
    - $245.42  if  Store Mailing List
    +   $0.00  if  Credit Card Only                   (reference level)
```

### What each term means to the business

| Term | Reading |
|---|---|
| **Intercept, $303.46** | The fitted baseline for a Credit-Card-Only customer purchasing zero products. Not a meaningful customer — it anchors the line, and should not be quoted as "a customer is worth $303". |
| **$66.98 per product** | Each additional product in the average basket is associated with about $67 more in average sale amount, holding segment constant. |
| **+$281.84 dual relationship** | A `Loyalty Club and Credit Card` customer is associated with $282 more per sale than an otherwise identical Credit-Card-Only customer. The highest-value relationship in the business. |
| **-$149.36 loyalty only** | Loyalty membership without a store card is associated with $149 *less* per sale. Loyalty programme membership alone is not a marker of high value. |
| **-$245.42 store list** | The weakest relationship, $245 below the reference. These customers spend least even after accounting for basket size. |

### Worked example

A `Loyalty Club and Credit Card` customer who averages 6 products:

```
303.46 + (66.98 x 6) + 281.84 = $987.18
```

Any stakeholder can reproduce that with a calculator. That transparency is
the reason for choosing this model.

**Note on causality.** These are associations, not causal effects. Upgrading a
customer into the dual-relationship segment would not automatically add $282 to
their basket — the segments differ in many ways beyond the label.

## 5. Model evaluation

Three metrics, because each answers a different question.

In [ ]:
metrics_table = pd.DataFrame([
    {"Metric": "R-squared", "Fitted": insample["r2"], "Held-out": holdout["test"]["r2"],
     "Question it answers": "How much of the variation in spend does the model explain?"},
    {"Metric": "MAE ($)", "Fitted": insample["mae"], "Held-out": holdout["test"]["mae"],
     "Question it answers": "How far off is a typical prediction, in dollars?"},
    {"Metric": "RMSE ($)", "Fitted": insample["rmse"], "Held-out": holdout["test"]["rmse"],
     "Question it answers": "How far off is it once large errors are penalised more heavily?"},
])
metrics_table

### What R-squared = 0.84 actually means

> The model explains approximately **84% of the variation** in average sale
> amount across the customers evaluated.

**It is not an accuracy rate.** It does not mean predictions are correct 84% of
the time, and it does not mean individual predictions are within 16% of the
truth. R-squared is a measure of *explained variance*: it compares the model's
errors against the errors you would make by predicting the overall mean for
everyone. A model can explain most of the variance and still be meaningfully
wrong about any individual customer.

Calling it "84% accurate" is the single most common way this figure is
misrepresented, and it leads directly to over-confident spending decisions. This
repository prohibits that phrasing (BRule-10).

### What MAE tells you that R-squared does not

Held-out **MAE is $93.40**. A typical prediction is off by about $93 on a mean
predicted sale of $553 — roughly **17%** in either direction. That is the number
a manager should hold in mind.

**RMSE ($140.19) is noticeably higher than MAE ($93.40)**, which tells you the
errors are not uniformly sized: there are a minority of large misses, consistent
with the high-value outliers found in EDA.

### Translating error into campaign dollars

In [ ]:
band = ev.financial_error_band(holdout["test"]["mae"], n_customers=250, gross_margin=0.50)
print(f"Held-out MAE per customer:            ${band['mae']:,.2f}")
print(f"Customers in campaign:                 {band['customers']}")
print(f"Worst-case revenue swing:             ${band['worst_case_revenue_swing']:,.2f}")
print(f"Worst-case gross profit swing:        ${band['worst_case_gross_profit_swing']:,.2f}")
print()
print("Note: this deliberately assumes every prediction errs in the SAME direction.")
print("In practice over- and under-predictions partially offset, so the realistic")
print("band is materially narrower. It is framed pessimistically on purpose.")

## 6. Residual analysis

Residuals are what the model got wrong. Their pattern reveals where it can be
trusted and where it cannot.

In [ ]:
res = ev.residual_frame(y, fitted)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(res["Predicted"], res["Residual"], alpha=0.3, s=12, color="#2b6cb0")
axes[0].axhline(0, color="#c53030", linestyle="--")
axes[0].set_title("Residuals vs fitted values")
axes[0].set_xlabel("Predicted sale amount ($)"); axes[0].set_ylabel("Residual ($)")

axes[1].hist(res["Residual"], bins=50, color="#2f855a", edgecolor="white")
axes[1].axvline(0, color="#c53030", linestyle="--")
axes[1].set_title("Distribution of residuals")
axes[1].set_xlabel("Residual ($)"); axes[1].set_ylabel("Customers")

from scipy import stats
stats.probplot(res["Residual"], dist="norm", plot=axes[2])
axes[2].set_title("Normal Q-Q plot")

plt.tight_layout(); plt.show()

print(f"Mean residual:        ${res['Residual'].mean():,.4f}  (should be ~0)")
print(f"Residual std dev:     ${res['Residual'].std():,.2f}")
print(f"Largest under-predict: ${res['Residual'].max():,.2f}")
print(f"Largest over-predict:  ${res['Residual'].min():,.2f}")

### Business interpretation of the residuals

**What is fine.** Residuals centre on zero with no systematic bias — the model
does not consistently over- or under-forecast overall. That is what makes the
campaign-level total trustworthy even where individual predictions are not.

**What to watch.** Two departures from textbook behaviour:

1. **Fanning at the high end.** Spread widens as predicted values grow. The model
   is more reliable for low-value customers than high-value ones, which is the
   cost of retaining the high-value outliers identified in EDA.
2. **Heavier tails than normal.** The Q-Q plot bends at the extremes, meaning a
   minority of customers are badly mispredicted — almost all of them very
   high-spending ones the linear model pulls toward the middle.

**Why this is acceptable for this decision.** The campaign forecast is a
**sum across 250 customers**, not a set of 250 individual promises. Errors
in opposite directions offset in aggregate, which is why the total is far more
reliable than any single prediction.

**Where it would not be acceptable.** If the campaign were selecting a small
number of customers for expensive individual treatment — a high-touch sales call,
say — this error structure would matter a great deal, because the mistakes
concentrate exactly among the high-value customers such a programme would target.

## 7. Benchmark against more flexible models

A sanity check on model choice, not a leaderboard. The question is whether
interpretability is costing anything material.

In [ ]:
comparison = md.compare_alternatives(customers)
comparison

### Interpretation and model selection decision

| Model | Test R-squared | Test MAE |
|---|---|---|
| Linear Regression (baseline) | 0.832 | $93.40 |
| Decision Tree (depth 5) | 0.879 | $79.13 |
| Random Forest (200 trees) | 0.879 | $78.76 |

The flexible models **do** perform better — about 4.7 points of additional
explained variance and roughly $15 less error per customer. That is a real
improvement and it should be reported honestly rather than buried.

**The linear model is nonetheless retained.** The reasoning:

1. **The improvement is immaterial to the decision.** $15 per customer across 250
   customers is about $3,750 of revenue and $1,875 of gross profit, against an
   expected net profit of $21,987. It does not come close to changing whether the
   campaign should run.
2. **It is dwarfed by assumption uncertainty.** Moving the gross margin from 50%
   to 45% changes net profit by about $2,400 — more than the entire model
   upgrade. The binding constraint on this forecast is the quality of the
   business assumptions, not the sophistication of the estimator.
3. **Interpretability has direct business value here.** The deliverable is not
   only a set of predictions; it is an answer to "what drives customer value",
   which feeds list-building and segment strategy. A coefficient of $281.84 is
   usable in a marketing meeting. A feature-importance score is not.
4. **Auditability.** Finance can reproduce any prediction by hand.

**When this decision should be revisited.** If the model moves from campaign
decision support into per-customer automated targeting at scale, where accumulated
per-customer error starts to matter more than explanation, the tree-based model
becomes the better choice.

This is exactly the trade-off a business analyst should make explicitly rather
than defaulting to whichever model scores highest.

## 8. Model limitations

| # | Limitation | Consequence |
|---|---|---|
| 1 | **Two predictors only.** No recency, frequency, product category, channel or seasonality data exists. | 16% of the variation in spend is unexplained by anything available. |
| 2 | **Under-predicts high-value customers.** Linear fit pulls extreme values toward the centre. | The most profitable prospects are likely worth *more* than the model says. The forecast is conservative at the top. |
| 3 | **Association, not causation.** Coefficients describe patterns, not levers. | Moving a customer into a higher segment would not automatically add $282 to their basket. |
| 4 | **Partly arithmetic.** Products purchased and sale amount are both averages over the same history. | Part of the fit restates an accounting relationship rather than discovering a behavioural driver. |
| 5 | **Population mismatch.** The campaign list skews towards segments that are a smaller share of the training data (C-04). | Forecast leans on the `Loyalty Club and Credit Card` coefficient, estimated from 194 records. |
| 6 | **No timestamp.** The age of the training data is unknown (C-09). | Drift cannot be assessed against elapsed time. |
| 7 | **Single geography.** `State` is effectively constant. | Do not generalise to other markets without revalidation. |
| 8 | **Predicts spend, not response.** Response probability comes from a model not supplied here (A-05). | Half of the expected-revenue calculation rests on an input this project cannot validate. |

## Summary

A two-predictor linear model explains approximately **84% of the variation** in
average sale amount, with a held-out mean absolute error of **$93.40** per
customer. It generalises cleanly (test R-squared 0.832 against train 0.839) and
its coefficients are directly interpretable in dollars.

More flexible models perform modestly better but the gain is immaterial against
the uncertainty in the business assumptions, and the interpretability is worth
more than the accuracy.

Proceed to [`03_campaign_profitability.ipynb`](03_campaign_profitability.ipynb)
to convert these predictions into campaign economics.